# Inference Lab: vLLM / SGLang · Kaggle T4

**Первый кейс — autoregressive LLM serving**, не video inference. Отдельный видеокейс:
SGLang Diffusion / vLLM-Omni / FastVideo; K6 baseline находится в основном репозитории.

GPU **T4 ×2**, Internet ON. Для первого сравнения используется **только GPU 0**;
движки запускаются последовательно. Модель: Qwen2.5-0.5B-Instruct, общие веса FP16.
Среды движков изолированы от Python kernel. Никаких изменений ComfyUI или основного курса.

Полный запуск устанавливает CUDA-зависимости и расходует GPU quota. Это exploratory benchmark:
32 запроса на ячейку, три независимых перезапуска, concurrency 1/2/4/8/16.
Для устойчивого tail latency увеличьте число запросов после первого успешного baseline.
Численные результаты не заявлены до выполнения ячеек.


## 1. Проверить оборудование до установки

T4: 16 GB на устройство, а не единый 32 GB пул. В этом прогоне FP16; нативные BF16/FP8
Tensor Core операции не исследуются. Фактические driver, CUDA runtime и GPU записываются в Output.


In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys
if not Path('/kaggle/working').is_dir() or not shutil.which('nvidia-smi'):
    raise RuntimeError('Use Kaggle with GPU T4 ×2. No installation or weights downloaded.')
print(subprocess.check_output(['nvidia-smi', '--query-gpu=index,name,memory.total,driver_version,compute_cap',
                               '--format=csv,noheader'], text=True))
print('Scratch free GiB:', round(shutil.disk_usage('/kaggle/working').free / 2**30, 1))


## 2. Восстановить закреплённые исходники лаборатории

Самодостаточная копия пяти Python-файлов. Их SHA-256 хранится в metadata notebook;
исходники сохраняются рядом с результатами. Эта ячейка только записывает код в scratch.
В репозитории редактируйте `.py`, затем пересоберите notebook генератором.


In [ ]:
SOURCES = {'harness.py': '"""Closed-loop OpenAI chat benchmark; stdlib transport, no inference dependencies.\n'
               '\n'
               'Client TTFT is the first non-empty text delta. SSE chunk gaps are NOT token ITL.\n'
               'Only server-reported usage is used for token throughput. No retry of timed requests.\n'
               '"""\n'
               'import argparse\n'
               'import concurrent.futures\n'
               'import http.client\n'
               'import json\n'
               'import math\n'
               'import queue\n'
               'import random\n'
               'import shutil\n'
               'import subprocess\n'
               'import threading\n'
               'import time\n'
               'import urllib.parse\n'
               'import urllib.request\n'
               'from datetime import datetime, timezone\n'
               'from pathlib import Path\n'
               '\n'
               '\n'
               'def write_json(path, value):\n'
               '    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\\n")\n'
               '\n'
               '\n'
               'def percentile(values, fraction):\n'
               '    values = sorted(value for value in values if value is not None)\n'
               '    if not values:\n'
               '        return None\n'
               '    position = (len(values) - 1) * fraction\n'
               '    lower = math.floor(position)\n'
               '    upper = math.ceil(position)\n'
               '    return values[lower] + (values[upper] - values[lower]) * (position - lower)\n'
               '\n'
               '\n'
               'def connection_for(base_url):\n'
               '    url = urllib.parse.urlsplit(base_url)\n'
               '    if url.scheme not in ("http", "https") or not url.hostname:\n'
               '        raise ValueError("Expected an http(s) base URL")\n'
               '    cls = http.client.HTTPSConnection if url.scheme == "https" else '
               'http.client.HTTPConnection\n'
               '    return cls(url.hostname, url.port, timeout=180), url.path.rstrip("/")\n'
               '\n'
               '\n'
               'def sse_events(response):\n'
               '    """Handle multi-line SSE data and CRLF; HTTP framing is handled by http.client."""\n'
               '    parts = []\n'
               '    while True:\n'
               '        line = response.readline()\n'
               '        if not line:\n'
               '            if parts:\n'
               '                yield "\\n".join(parts)\n'
               '            return\n'
               '        line = line.decode("utf-8").rstrip("\\r\\n")\n'
               '        if not line:\n'
               '            if parts:\n'
               '                yield "\\n".join(parts)\n'
               '                parts = []\n'
               '        elif line.startswith("data:"):\n'
               '            parts.append(line[5:].lstrip(" "))\n'
               '\n'
               '\n'
               'def perform_request(connection, api_path, item, model, *, stream=True, ignore_eos=True):\n'
               '    payload = dict(model=model, messages=item["messages"], '
               'max_tokens=item["output_tokens"],\n'
               '                   temperature=0, top_p=1, presence_penalty=0, frequency_penalty=0,\n'
               '                   seed=42, stream=stream, ignore_eos=ignore_eos)\n'
               '    if stream:\n'
               '        payload["stream_options"] = {"include_usage": True}\n'
               '    started = time.perf_counter()\n'
               '    row = dict(id=item["id"], workload=item.get("workload"), success=False, text="",\n'
               '               expected_prompt_tokens=item.get("prompt_tokens"),\n'
               '               requested_output_tokens=item["output_tokens"], text_event_offsets_s=[],\n'
               '               usage=None, finish_reason=None, error=None, itl_ms=None)\n'
               '    try:\n'
               '        connection.request("POST", api_path + "/chat/completions",\n'
               '                           body=json.dumps(payload).encode(),\n'
               '                           headers={"Content-Type": "application/json"})\n'
               '        response = connection.getresponse()\n'
               '        if response.status != 200:\n'
               '            raise RuntimeError(f"HTTP {response.status}: '
               '{response.read(4096).decode(errors=\'replace\')}")\n'
               '        if stream:\n'
               '            done = False\n'
               '            for data in sse_events(response):\n'
               '                if data.strip() == "[DONE]":\n'
               '                    done = True\n'
               '                    # Drain the HTTP body before reusing the persistent connection.\n'
               '                    response.read()\n'
               '                    break\n'
               '                event = json.loads(data)\n'
               '                if event.get("error"):\n'
               '                    raise RuntimeError(str(event["error"]))\n'
               '                if event.get("usage"):\n'
               '                    row["usage"] = event["usage"]\n'
               '                for choice in event.get("choices", []):\n'
               '                    delta = choice.get("delta", {})\n'
               '                    text_delta = delta.get("content")\n'
               '                    if text_delta:\n'
               '                        row["text"] += text_delta\n'
               '                        row["text_event_offsets_s"].append(time.perf_counter() - started)\n'
               '                    if choice.get("finish_reason"):\n'
               '                        row["finish_reason"] = choice["finish_reason"]\n'
               '            if not done:\n'
               '                raise RuntimeError("Truncated SSE: missing [DONE]")\n'
               '        else:\n'
               '            event = json.loads(response.read())\n'
               '            row["usage"] = event.get("usage")\n'
               '            row["text"] = event["choices"][0]["message"].get("content") or ""\n'
               '            row["finish_reason"] = event["choices"][0].get("finish_reason")\n'
               '        if not row["text"].strip():\n'
               '            raise RuntimeError("No generated text")\n'
               '        usage = row["usage"] or {}\n'
               '        for key in ("prompt_tokens", "completion_tokens"):\n'
               '            if not isinstance(usage.get(key), int) or usage[key] <= 0:\n'
               '                raise RuntimeError(f"Missing/invalid actual usage.{key}; no token estimate '
               'substituted")\n'
               '        expected = item.get("prompt_tokens")\n'
               '        if expected is not None and usage["prompt_tokens"] != expected:\n'
               '            raise RuntimeError(f"Chat-template/tokenizer mismatch: expected {expected}, got '
               '{usage[\'prompt_tokens\']}")\n'
               '        if ignore_eos and usage["completion_tokens"] != item["output_tokens"]:\n'
               '            raise RuntimeError("Forced-length request returned a different token count")\n'
               '        row["success"] = True\n'
               '    except (OSError, ValueError, KeyError, IndexError, RuntimeError, '
               'http.client.HTTPException) as error:\n'
               '        row["error"] = f"{type(error).__name__}: {error}"\n'
               '        connection.close()\n'
               '    row["e2e_ms"] = (time.perf_counter() - started) * 1000\n'
               '    offsets = row["text_event_offsets_s"]\n'
               '    row["ttft_ms"] = offsets[0] * 1000 if offsets else None\n'
               '    count = (row["usage"] or {}).get("completion_tokens", 0)\n'
               '    row["tpot_ms"] = ((offsets[-1] - offsets[0]) * 1000 / (count - 1)\n'
               '                      if count > 1 and len(offsets) > 1 else None)\n'
               '    row["inter_chunk_ms"] = [(right - left) * 1000 for left, right in zip(offsets, '
               'offsets[1:])]\n'
               '    row["text_chunks"] = len(offsets)\n'
               '    return row\n'
               '\n'
               '\n'
               'def smoke_server(base_url, model, output):\n'
               '    with urllib.request.urlopen(base_url + "/models", timeout=10) as response:\n'
               '        available = json.load(response)\n'
               '    if model not in [item["id"] for item in available.get("data", [])]:\n'
               '        raise RuntimeError("Served model name missing from /v1/models")\n'
               '    item = dict(id="semantic-smoke", messages=[{"role": "user", "content":\n'
               '                "What is the capital of France? Answer with the city name only."}], '
               'output_tokens=24)\n'
               '    rows = []\n'
               '    for stream in (False, True):\n'
               '        connection, path = connection_for(base_url)\n'
               '        try:\n'
               '            row = perform_request(connection, path, item, model, stream=stream, '
               'ignore_eos=False)\n'
               '            row["streaming"] = stream\n'
               '            rows.append(row)\n'
               '        finally:\n'
               '            connection.close()\n'
               '    write_json(output, rows)\n'
               '    if not all(row["success"] and "paris" in row["text"].lower() for row in rows):\n'
               '        raise RuntimeError(f"Semantic or streaming smoke failed; inspect {output}")\n'
               '\n'
               '\n'
               'def request_batch(base_url, model, items, concurrency):\n'
               '    pending = queue.Queue()\n'
               '    for item in items:\n'
               '        pending.put(item)\n'
               '    rows = []\n'
               '    lock = threading.Lock()\n'
               '\n'
               '    def worker():\n'
               '        connection, path = connection_for(base_url)\n'
               '        try:\n'
               '            while True:\n'
               '                try:\n'
               '                    item = pending.get_nowait()\n'
               '                except queue.Empty:\n'
               '                    return\n'
               '                row = perform_request(connection, path, item, model)\n'
               '                with lock:\n'
               '                    rows.append(row)\n'
               '        finally:\n'
               '            connection.close()\n'
               '\n'
               '    started = time.perf_counter()\n'
               '    with concurrent.futures.ThreadPoolExecutor(max_workers=concurrency) as pool:\n'
               '        futures = [pool.submit(worker) for _ in range(concurrency)]\n'
               '        for future in futures:\n'
               '            future.result()\n'
               '    return rows, time.perf_counter() - started\n'
               '\n'
               '\n'
               'def summarize_rows(rows, wall_seconds, slo):\n'
               '    successful = [row for row in rows if row["success"]]\n'
               '    prompt_tokens = sum(row["usage"]["prompt_tokens"] for row in successful)\n'
               '    output_tokens = sum(row["usage"]["completion_tokens"] for row in successful)\n'
               '    good = [row for row in successful if row["ttft_ms"] is not None\n'
               '            and row["tpot_ms"] is not None and row["ttft_ms"] <= slo["ttft_ms"]\n'
               '            and row["tpot_ms"] <= slo["tpot_ms"] and row["e2e_ms"] <= slo["e2e_ms"]]\n'
               '    summary = dict(attempted=len(rows), successful=len(successful), '
               'wall_seconds=wall_seconds,\n'
               '                   error_rate=(len(rows) - len(successful)) / len(rows) if rows else None,\n'
               '                   request_throughput=len(successful) / wall_seconds,\n'
               '                   output_tokens_per_second=output_tokens / wall_seconds,\n'
               '                   total_tokens_per_second=(prompt_tokens + output_tokens) / wall_seconds,\n'
               '                   goodput_requests_per_second=len(good) / wall_seconds,\n'
               '                   actual_prompt_tokens=prompt_tokens, actual_output_tokens=output_tokens,\n'
               '                   p99_small_sample=len(successful) < 100,\n'
               '                   token_itl_available=False)\n'
               '    for key in ("ttft_ms", "tpot_ms", "e2e_ms"):\n'
               '        values = [row[key] for row in successful]\n'
               '        summary[key] = {f"p{p}": percentile(values, p / 100) for p in (50, 95, 99)}\n'
               '    return summary\n'
               '\n'
               '\n'
               'def resource_sample(server_pid=None):\n'
               '    sample = dict(monotonic_s=time.perf_counter(), gpu=[])\n'
               '    if shutil.which("nvidia-smi"):\n'
               '        result = subprocess.run([\n'
               '            "nvidia-smi", '
               '"--query-gpu=index,name,memory.used,memory.total,utilization.gpu,power.draw",\n'
               '            "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=5)\n'
               '        if result.returncode == 0:\n'
               '            for line in result.stdout.splitlines():\n'
               '                fields = [part.strip() for part in line.split(",")]\n'
               '                if len(fields) == 6:\n'
               '                    sample["gpu"].append(dict(zip(\n'
               '                        ("index", "name", "memory_used_mib", "memory_total_mib", '
               '"utilization_pct", "power_w"), fields)))\n'
               '    try:\n'
               '        import psutil\n'
               '        sample["node_ram_used_bytes"] = psutil.virtual_memory().used\n'
               '        if server_pid:\n'
               '            process = psutil.Process(server_pid)\n'
               '            processes = [process] + process.children(recursive=True)\n'
               '            # RSS sum double-counts shared pages; it is not unique physical memory.\n'
               '            sample["server_tree_rss_bytes"] = sum(p.memory_info().rss for p in processes if '
               'p.is_running())\n'
               '    except Exception as error:\n'
               '        # Monitoring may lose a process during shutdown; timed HTTP results remain valid.\n'
               '        sample["ram_error"] = type(error).__name__\n'
               '    return sample\n'
               '\n'
               '\n'
               'def metrics_snapshot(base_url, path):\n'
               '    url = urllib.parse.urlsplit(base_url)\n'
               '    try:\n'
               '        with urllib.request.urlopen(f"{url.scheme}://{url.netloc}/metrics", timeout=5) as '
               'response:\n'
               '            Path(path).write_bytes(response.read())\n'
               '    except OSError as error:\n'
               '        Path(path).write_text(f"# unavailable: {type(error).__name__}\\n")\n'
               '\n'
               '\n'
               'def run_benchmark(base_url, model, items, concurrency, output, *, labels=None,\n'
               '                  warmup=None, seed=42, server_pid=None,\n'
               '                  slo=None):\n'
               '    output = Path(output)\n'
               '    output.mkdir(parents=True, exist_ok=False)\n'
               '    if not items or concurrency < 1:\n'
               '        raise ValueError("Non-empty workload and positive concurrency required")\n'
               '    slo = slo or dict(ttft_ms=1000, tpot_ms=100, e2e_ms=30000)\n'
               '    write_json(output / "config.json", dict(base_url=base_url, model=model, '
               'concurrency=concurrency,\n'
               '               seed=seed, labels=labels, slo=slo, load_model="closed_loop", streaming=True,\n'
               '               generation=dict(temperature=0, top_p=1, seed=42, ignore_eos=True)))\n'
               '    if warmup:\n'
               '        warm_rows, _ = request_batch(base_url, model, warmup, min(concurrency, '
               'len(warmup)))\n'
               '        write_json(output / "warmup.json", warm_rows)\n'
               '        if not all(row["success"] for row in warm_rows):\n'
               '            raise RuntimeError(f"Warmup failed; inspect {output / \'warmup.json\'}")\n'
               '    metrics_snapshot(base_url, output / "metrics-before.prom")\n'
               '    resources = []\n'
               '    stop = threading.Event()\n'
               '\n'
               '    def monitor():\n'
               '        while not stop.is_set():\n'
               '            try:\n'
               '                resources.append(resource_sample(server_pid))\n'
               '            except (OSError, subprocess.TimeoutExpired) as error:\n'
               '                resources.append(dict(error=str(error)))\n'
               '            stop.wait(0.5)\n'
               '\n'
               '    thread = threading.Thread(target=monitor, daemon=True)\n'
               '    ordered = list(items)\n'
               '    random.Random(seed).shuffle(ordered)\n'
               '    thread.start()\n'
               '    try:\n'
               '        rows, wall = request_batch(base_url, model, ordered, concurrency)\n'
               '    finally:\n'
               '        stop.set()\n'
               '        thread.join(timeout=6)\n'
               '    (output / "requests.jsonl").write_text("".join(json.dumps(row, ensure_ascii=False) + '
               '"\\n" for row in rows))\n'
               '    (output / "resources.jsonl").write_text("".join(json.dumps(row) + "\\n" for row in '
               'resources))\n'
               '    metrics_snapshot(base_url, output / "metrics-after.prom")\n'
               '    summary = summarize_rows(rows, wall, slo)\n'
               '    summary.update(labels or {})\n'
               '    summary.update(concurrency=concurrency, '
               'timestamp_utc=datetime.now(timezone.utc).isoformat())\n'
               '    write_json(output / "summary.json", summary)\n'
               '    return summary\n'
               '\n'
               '\n'
               'if __name__ == "__main__":\n'
               '    parser = argparse.ArgumentParser(description=__doc__)\n'
               '    parser.add_argument("--base-url", required=True)\n'
               '    parser.add_argument("--model", required=True)\n'
               '    parser.add_argument("--workload", required=True, type=Path)\n'
               '    parser.add_argument("--concurrency", type=int, default=1)\n'
               '    parser.add_argument("--output", required=True, type=Path)\n'
               '    args = parser.parse_args()\n'
               '    items = [json.loads(line) for line in args.workload.read_text().splitlines() if line]\n'
               '    result = run_benchmark(args.base_url, args.model, items, args.concurrency, args.output)\n'
               '    print(json.dumps(result, indent=2))\n',
 'discover_hardware.py': '"""Read-only environment inventory. No installation or model downloads."""\n'
                         'import argparse\n'
                         'import importlib.metadata\n'
                         'import json\n'
                         'import os\n'
                         'import platform\n'
                         'import shutil\n'
                         'import subprocess\n'
                         'from datetime import datetime, timezone\n'
                         'from pathlib import Path\n'
                         '\n'
                         '\n'
                         'def command_output(argv):\n'
                         '    try:\n'
                         '        result = subprocess.run(argv, capture_output=True, text=True, timeout=30)\n'
                         '        return {"returncode": result.returncode, "stdout": result.stdout.strip(),\n'
                         '                "stderr": result.stderr.strip()}\n'
                         '    except (OSError, subprocess.TimeoutExpired) as error:\n'
                         '        return {"error": str(error)}\n'
                         '\n'
                         '\n'
                         'def discover_hardware():\n'
                         '    disk = shutil.disk_usage(Path.cwd())\n'
                         '    result = dict(timestamp_utc=datetime.now(timezone.utc).isoformat(),\n'
                         '                  system=platform.system(), release=platform.release(),\n'
                         '                  architecture=platform.machine(), '
                         'python=platform.python_version(),\n'
                         '                  disk_free_gib=round(disk.free / 2**30, 2), packages={})\n'
                         '    result["cpu_count"] = os.cpu_count()\n'
                         '    try:\n'
                         '        result["physical_ram_bytes"] = os.sysconf("SC_PHYS_PAGES") * '
                         'os.sysconf("SC_PAGE_SIZE")\n'
                         '    except (OSError, ValueError):\n'
                         '        result["physical_ram_bytes"] = None\n'
                         '    if platform.system() == "Linux":\n'
                         '        result["cgroup"] = {}\n'
                         '        for name in ("memory.max", "memory.events", "memory.peak"):\n'
                         '            path = Path("/sys/fs/cgroup") / name\n'
                         '            if path.is_file():\n'
                         '                result["cgroup"][name] = path.read_text().strip()\n'
                         '    for package in ("torch", "mlx", "mlx-lm", "vllm", "vllm-metal", "sglang",\n'
                         '                    "transformers", "psutil"):\n'
                         '        try:\n'
                         '            result["packages"][package] = importlib.metadata.version(package)\n'
                         '        except importlib.metadata.PackageNotFoundError:\n'
                         '            result["packages"][package] = None\n'
                         '    if platform.system() == "Darwin":\n'
                         '        # Select only hardware fields; never persist serial numbers or machine '
                         'UUIDs.\n'
                         '        raw = command_output(["system_profiler", "SPHardwareDataType", "-json"])\n'
                         '        try:\n'
                         '            hardware = json.loads(raw["stdout"])["SPHardwareDataType"][0]\n'
                         '            result["hardware"] = {key: hardware.get(key) for key in\n'
                         '                                  ("chip_type", "physical_memory", '
                         '"number_processors")}\n'
                         '        except (KeyError, ValueError):\n'
                         '            result["hardware"] = {"error": raw.get("error", raw.get("stderr"))}\n'
                         '        result["display"] = command_output(["system_profiler", '
                         '"SPDisplaysDataType"])\n'
                         '        result["macos"] = platform.mac_ver()[0]\n'
                         '    if shutil.which("nvidia-smi"):\n'
                         '        result["nvidia"] = command_output([\n'
                         '            "nvidia-smi", '
                         '"--query-gpu=index,name,memory.total,driver_version,compute_cap",\n'
                         '            "--format=csv,noheader,nounits"])\n'
                         '        result["topology"] = command_output(["nvidia-smi", "topo", "-m"])\n'
                         '    if result["packages"]["torch"]:\n'
                         '        import torch\n'
                         '        result["torch_devices"] = dict(cuda=torch.cuda.is_available(),\n'
                         '                                       cuda_runtime=torch.version.cuda,\n'
                         '                                       mps_built=torch.backends.mps.is_built(),\n'
                         '                                       '
                         'mps_available_in_this_process=torch.backends.mps.is_available())\n'
                         '    return result\n'
                         '\n'
                         '\n'
                         'if __name__ == "__main__":\n'
                         '    parser = argparse.ArgumentParser(description=__doc__)\n'
                         '    parser.add_argument("--output", type=Path)\n'
                         '    args = parser.parse_args()\n'
                         '    result = discover_hardware()\n'
                         '    encoded = json.dumps(result, ensure_ascii=False, indent=2) + "\\n"\n'
                         '    if args.output:\n'
                         '        args.output.parent.mkdir(parents=True, exist_ok=True)\n'
                         '        args.output.write_text(encoded)\n'
                         '    print(encoded)\n',
 'prepare_workloads.py': '"""Run inside an engine environment: pin one snapshot and materialize chat '
                         'inputs."""\n'
                         'import argparse\n'
                         'import hashlib\n'
                         'import json\n'
                         'from pathlib import Path\n'
                         '\n'
                         '\n'
                         'def prepare_workloads(model_id, revision, cache_dir, output, requests, '
                         'warmup_requests=4,\n'
                         '                      warmup_input_tokens=512, warmup_output_tokens=64):\n'
                         '    from huggingface_hub import snapshot_download\n'
                         '    from transformers import AutoTokenizer\n'
                         '    output = Path(output)\n'
                         '    output.mkdir(parents=True, exist_ok=True)\n'
                         '    snapshot = Path(snapshot_download(model_id, revision=revision, '
                         'cache_dir=cache_dir,\n'
                         '                                      ignore_patterns=["*.bin", "*.h5", '
                         '"*.msgpack", "*.onnx"]))\n'
                         '    tokenizer = AutoTokenizer.from_pretrained(snapshot, trust_remote_code=False)\n'
                         '\n'
                         '    def item(kind, index, target, output_tokens, shared=False):\n'
                         '        identity = f"Request {index:06d}. "\n'
                         '        question = " Explain inference optimization as a numbered list."\n'
                         '        prefix = "Read the following context. " if shared else identity + "Read '
                         'the context. "\n'
                         '        suffix = (identity if shared else "") + question\n'
                         '        padding = target\n'
                         '        for _ in range(12):\n'
                         '            messages = [{"role": "user", "content": prefix + " data" * max(0, '
                         'padding) + suffix}]\n'
                         '            ids = tokenizer.apply_chat_template(messages, tokenize=True, '
                         'add_generation_prompt=True,\n'
                         '                                                 return_dict=False)\n'
                         '            if not isinstance(ids, list) or not all(isinstance(token, int) for '
                         'token in ids):\n'
                         '                raise TypeError("Expected a flat token ID list from explicit '
                         'return_dict=False")\n'
                         '            if len(ids) == target:\n'
                         '                return dict(id=f"{kind}-{index:06d}", workload=kind, '
                         'messages=messages,\n'
                         '                            prompt_tokens=len(ids), output_tokens=output_tokens)\n'
                         '            padding += target - len(ids)\n'
                         '        raise RuntimeError(f"Cannot construct exact {target}-token chat input: got '
                         '{len(ids)}")\n'
                         '\n'
                         '    workloads = {}\n'
                         '    for kind, length, generated in (("short", 128, 64), ("standard", 512, 128), '
                         '("long", 2048, 256)):\n'
                         '        workloads[kind] = [item(kind, i, length, generated) for i in '
                         'range(requests)]\n'
                         '    workloads["mixed"] = [item("mixed", i, 128 if i % 2 == 0 else 2048,\n'
                         '                               64 if i % 2 == 0 else 256) for i in '
                         'range(requests)]\n'
                         '    workloads["shared"] = [item("shared", i, 2048, 256, True) for i in '
                         'range(requests)]\n'
                         '    workloads["warmup"] = [item("warmup", 100000 + i, warmup_input_tokens, '
                         'warmup_output_tokens)\n'
                         '                           for i in range(warmup_requests)]\n'
                         '    for kind, rows in workloads.items():\n'
                         '        (output / f"{kind}.jsonl").write_text("".join(json.dumps(row) + "\\n" for '
                         'row in rows))\n'
                         '    files = {}\n'
                         '    for path in sorted(snapshot.rglob("*")):\n'
                         '        if path.is_file():\n'
                         '            digest = hashlib.sha256()\n'
                         '            with path.open("rb") as handle:\n'
                         '                for block in iter(lambda: handle.read(1024 * 1024), b""):\n'
                         '                    digest.update(block)\n'
                         '            files[str(path.relative_to(snapshot))] = '
                         'dict(bytes=path.stat().st_size,\n'
                         '                                                        '
                         'sha256=digest.hexdigest())\n'
                         '    manifest = dict(model_id=model_id, requested_revision=revision, '
                         'resolved_revision=snapshot.name,\n'
                         '                    snapshot_path=str(snapshot), files=files,\n'
                         '                    '
                         'chat_template_sha256=hashlib.sha256(tokenizer.chat_template.encode()).hexdigest(),\n'
                         '                    synthetic=True, '
                         'exact_prompt_lengths_include_chat_template=True)\n'
                         '    (output / "model.json").write_text(json.dumps(manifest, indent=2) + "\\n")\n'
                         '    return manifest\n'
                         '\n'
                         '\n'
                         'if __name__ == "__main__":\n'
                         '    parser = argparse.ArgumentParser(description=__doc__)\n'
                         '    parser.add_argument("--model", required=True)\n'
                         '    parser.add_argument("--revision", required=True)\n'
                         '    parser.add_argument("--cache", required=True)\n'
                         '    parser.add_argument("--output", required=True)\n'
                         '    parser.add_argument("--requests", type=int, default=32)\n'
                         '    parser.add_argument("--warmup-requests", type=int, default=4)\n'
                         '    parser.add_argument("--warmup-input-tokens", type=int, default=512)\n'
                         '    parser.add_argument("--warmup-output-tokens", type=int, default=64)\n'
                         '    args = parser.parse_args()\n'
                         '    prepare_workloads(args.model, args.revision, args.cache, args.output, '
                         'args.requests,\n'
                         '                      args.warmup_requests, args.warmup_input_tokens, '
                         'args.warmup_output_tokens)\n',
 'run_kaggle.py': '"""Sequential CUDA campaign. Engine installations never modify the notebook kernel."""\n'
                  'import argparse\n'
                  'import hashlib\n'
                  'import json\n'
                  'import os\n'
                  'import random\n'
                  'import shutil\n'
                  'import signal\n'
                  'import subprocess\n'
                  'import sys\n'
                  'import time\n'
                  'import urllib.request\n'
                  'from pathlib import Path\n'
                  '\n'
                  '\n'
                  'DEFAULT_CONFIG = dict(model="Qwen/Qwen2.5-0.5B-Instruct", '
                  'revision="7ae557604adf67be50417f59c2c2f167def9a775",\n'
                  '                      engines={"vllm": "0.31.0", "sglang": "0.5.22"},\n'
                  '                      requests=32, repeats=3, concurrency=[1, 2, 4, 8, 16], seed=42,\n'
                  '                      tensor_parallel=1, gpu_devices="0", memory_fraction=0.70,\n'
                  '                      max_model_len=4096, max_sequences=32,\n'
                  '                      arms=["baseline", "chunked4096", "chunked512", "prefix"],\n'
                  '                      slo=dict(ttft_ms=1000, tpot_ms=100, e2e_ms=30000))\n'
                  '\n'
                  '\n'
                  'def server_command(engine, python, model_path, config, arm, port):\n'
                  '    chunked = arm in ("chunked4096", "chunked512")\n'
                  '    prefix = arm == "prefix"\n'
                  '    if engine == "vllm":\n'
                  '        return [str(python), "-m", "vllm.entrypoints.openai.api_server",\n'
                  '                "--model", str(model_path), "--served-model-name", "qwen-lab",\n'
                  '                "--host", "127.0.0.1", "--port", str(port), "--dtype", "float16",\n'
                  '                "--max-model-len", str(config["max_model_len"]),\n'
                  '                "--max-num-seqs", str(config["max_sequences"]),\n'
                  '                "--gpu-memory-utilization", str(config["memory_fraction"]),\n'
                  '                "--tensor-parallel-size", str(config["tensor_parallel"]),\n'
                  '                "--seed", str(config["seed"]), "--generation-config", "vllm",\n'
                  '                "--attention-backend", "TRITON_ATTN", "--enforce-eager",\n'
                  '                "--enable-prefix-caching" if prefix else "--no-enable-prefix-caching",\n'
                  '                "--enable-chunked-prefill" if chunked else '
                  '"--no-enable-chunked-prefill",\n'
                  '                "--max-num-batched-tokens", "512" if arm == "chunked512" else "4096"]\n'
                  '    if engine == "sglang":\n'
                  '        argv = [str(python), "-m", "sglang.launch_server", "--model-path", '
                  'str(model_path),\n'
                  '                "--served-model-name", "qwen-lab", "--host", "127.0.0.1", "--port", '
                  'str(port),\n'
                  '                "--dtype", "float16", "--context-length", str(config["max_model_len"]),\n'
                  '                "--tp-size", str(config["tensor_parallel"]),\n'
                  '                "--mem-fraction-static", str(config["memory_fraction"]),\n'
                  '                "--max-running-requests", str(config["max_sequences"]),\n'
                  '                "--attention-backend", "triton", "--sampling-backend", "pytorch",\n'
                  '                "--disable-cuda-graph", "--random-seed", str(config["seed"]),\n'
                  '                "--schedule-policy", "fcfs", "--chunked-prefill-size",\n'
                  '                ("512" if arm == "chunked512" else "4096") if chunked else "-1",\n'
                  '                "--max-prefill-tokens", "4096", "--enable-metrics"]\n'
                  '        if not prefix:\n'
                  '            argv.append("--disable-radix-cache")\n'
                  '        if config.get("sglang_max_total_tokens"):\n'
                  '            argv += ["--max-total-tokens", str(config["sglang_max_total_tokens"])]\n'
                  '        return argv\n'
                  '    raise ValueError(f"Unknown engine: {engine}")\n'
                  '\n'
                  '\n'
                  'def campaign_cells(arm, config):\n'
                  '    if config.get("smoke_only"):\n'
                  '        return [("short", 1)]\n'
                  '    if config.get("cells"):\n'
                  '        return [(kind, int(concurrency)) for kind, concurrency in config["cells"][arm]]\n'
                  '    if arm == "baseline":\n'
                  '        return [(kind, c) for kind in ("short", "standard", "long") for c in '
                  'config["concurrency"]] + [\n'
                  '            ("mixed", 8), ("mixed", 16), ("shared", 8)]\n'
                  '    if arm in ("chunked4096", "chunked512"):\n'
                  '        return [("mixed", 8), ("mixed", 16)]\n'
                  '    if arm == "prefix":\n'
                  '        return [("shared", 8)]\n'
                  '    raise ValueError(arm)\n'
                  '\n'
                  '\n'
                  'def stop_server(process):\n'
                  '    if process.poll() is not None:\n'
                  '        return\n'
                  '    # Only the process group created by this campaign is stopped.\n'
                  '    os.killpg(process.pid, signal.SIGTERM)\n'
                  '    try:\n'
                  '        process.wait(timeout=20)\n'
                  '    except subprocess.TimeoutExpired:\n'
                  '        os.killpg(process.pid, signal.SIGKILL)\n'
                  '        process.wait(timeout=10)\n'
                  '\n'
                  '\n'
                  'def wait_for_server(process, base_url, timeout=600):\n'
                  '    deadline = time.monotonic() + timeout\n'
                  '    origin = base_url.rsplit("/v1", 1)[0]\n'
                  '    while time.monotonic() < deadline:\n'
                  '        if process.poll() is not None:\n'
                  '            raise RuntimeError(f"Server exited with code {process.returncode}; inspect '
                  'server.log")\n'
                  '        try:\n'
                  '            with urllib.request.urlopen(origin + "/health", timeout=2) as response:\n'
                  '                if response.status == 200:\n'
                  '                    return\n'
                  '        except OSError:\n'
                  '            pass\n'
                  '        time.sleep(1)\n'
                  '    raise RuntimeError("Server readiness timeout; inspect server.log")\n'
                  '\n'
                  '\n'
                  'def run_logged(argv, log, env=None, timeout=1800):\n'
                  '    with Path(log).open("w") as handle:\n'
                  '        result = subprocess.run(argv, stdout=handle, stderr=subprocess.STDOUT,\n'
                  '                                env=env, timeout=timeout)\n'
                  '    if result.returncode:\n'
                  '        tail = Path(log).read_text(errors="replace")[-3000:]\n'
                  '        raise RuntimeError(f"Command failed ({result.returncode}); {log}\\n{tail}")\n'
                  '\n'
                  '\n'
                  'def cuda_driver_linker_path(scratch, env):\n'
                  '    """Expose the real driver to JIT linkers without changing system CUDA libraries."""\n'
                  '    candidates = []\n'
                  '    ldconfig = shutil.which("ldconfig")\n'
                  '    if ldconfig:\n'
                  '        listing = subprocess.run([ldconfig, "-p"], capture_output=True, text=True, '
                  'timeout=10)\n'
                  '        for line in listing.stdout.splitlines():\n'
                  '            if "libcuda.so.1 " in line and "=>" in line:\n'
                  '                candidates.append(Path(line.rsplit("=>", 1)[1].strip()))\n'
                  '    candidates += [Path(path) for path in (\n'
                  '        "/usr/lib/x86_64-linux-gnu/libcuda.so.1", '
                  '"/usr/local/nvidia/lib64/libcuda.so.1",\n'
                  '        "/usr/lib64/libcuda.so.1")]\n'
                  '    driver = next((path.resolve() for path in candidates if path.is_file()), None)\n'
                  '    if driver is None:\n'
                  '        raise RuntimeError("Cannot locate the real libcuda.so.1 driver for the JIT '
                  'linker")\n'
                  '    link_dir = Path(scratch) / "cuda-driver-link"\n'
                  '    link_dir.mkdir(parents=True, exist_ok=True)\n'
                  '    link = link_dir / "libcuda.so"\n'
                  '    link.unlink(missing_ok=True)\n'
                  '    link.symlink_to(driver)\n'
                  '    previous = env.get("LIBRARY_PATH", "")\n'
                  '    env["LIBRARY_PATH"] = str(link_dir) + (os.pathsep + previous if previous else "")\n'
                  '    return dict(driver=str(driver), link=str(link), library_path=env["LIBRARY_PATH"],\n'
                  '                changes_system_libraries=False)\n'
                  '\n'
                  '\n'
                  'def run_campaign(config, output, benchmark_fn, smoke_fn, report_fn, source_dir):\n'
                  '    output, source_dir = Path(output), Path(source_dir)\n'
                  '    output.mkdir(parents=True, exist_ok=False)\n'
                  '    scratch = Path("/kaggle/temp/inference-lab")\n'
                  '    if not Path("/kaggle/working").is_dir() or not shutil.which("nvidia-smi"):\n'
                  '        raise RuntimeError("This runner requires Kaggle with NVIDIA GPU; no install '
                  'started")\n'
                  '    scratch.mkdir(parents=True, exist_ok=True)\n'
                  '    env = os.environ.copy()\n'
                  '    env.update(HF_HUB_DISABLE_TELEMETRY="1", HF_HUB_DISABLE_XET="1",\n'
                  '               TOKENIZERS_PARALLELISM="false", '
                  'CUDA_VISIBLE_DEVICES=config["gpu_devices"],\n'
                  '               UV_CACHE_DIR=str(scratch / "uv-cache"))\n'
                  '    # Do not copy credentials or the process environment into any artifact.\n'
                  '    (output / "campaign.json").write_text(json.dumps(config, indent=2) + "\\n")\n'
                  '    if config.get("cuda_driver_linker_fix", True):\n'
                  '        linker = cuda_driver_linker_path(scratch, env)\n'
                  '        (output / "cuda-linker.json").write_text(json.dumps(linker, indent=2) + "\\n")\n'
                  '        print("JIT linker uses driver:", linker["driver"], flush=True)\n'
                  '    source_copy = output / "source"\n'
                  '    shutil.copytree(source_dir, source_copy, ignore=shutil.ignore_patterns("__pycache__", '
                  '"*.pyc"))\n'
                  '    errors = []\n'
                  '    uv = shutil.which("uv")\n'
                  '    if not uv:\n'
                  '        bootstrap = scratch / "bootstrap"\n'
                  '        run_logged([sys.executable, "-m", "pip", "install", "--target", str(bootstrap),\n'
                  '                    "uv==0.9.5"], output / "bootstrap.log", env)\n'
                  '        uv = str(bootstrap / "bin/uv")\n'
                  '    engines = {}\n'
                  '    for engine, version in config["engines"].items():\n'
                  '        engine_out = output / engine\n'
                  '        engine_out.mkdir()\n'
                  '        python = scratch / "venvs" / engine / "bin/python"\n'
                  '        try:\n'
                  '            run_logged([uv, "venv", "--python", "3.12", "--seed", '
                  'str(python.parent.parent)],\n'
                  '                       engine_out / "venv.log", env)\n'
                  '            run_logged([uv, "pip", "install", "--python", str(python), '
                  '"--prerelease=allow",\n'
                  '                        f"{engine}=={version}"] + config.get("engine_constraints", '
                  '{}).get(engine, []),\n'
                  '                       engine_out / "install.log", env)\n'
                  '            run_logged([uv, "pip", "freeze", "--python", str(python)], engine_out / '
                  '"freeze.txt", env)\n'
                  '            run_logged([str(python), str(source_dir / "discover_hardware.py"),\n'
                  '                        "--output", str(engine_out / "environment.json")],\n'
                  '                       engine_out / "environment.log", env)\n'
                  '            engines[engine] = python\n'
                  '            print(f"Installed {engine} {version}", flush=True)\n'
                  '        except Exception as error:\n'
                  '            errors.append(dict(engine=engine, stage="install", error=str(error)))\n'
                  '            (output / "errors.json").write_text(json.dumps(errors, indent=2))\n'
                  '            print(f"{engine} install FAILED; continuing with other engine", flush=True)\n'
                  '    workloads = output / "workloads"\n'
                  '    if not engines:\n'
                  '        report_fn(output)\n'
                  '        return output\n'
                  '    prepare_python = next(iter(engines.values()))\n'
                  '    warmup = config.get("warmup", dict(requests=4, input_tokens=512, output_tokens=64))\n'
                  '    try:\n'
                  '        run_logged([str(prepare_python), str(source_dir / "prepare_workloads.py"),\n'
                  '                    "--model", config["model"], "--revision", config["revision"],\n'
                  '                    "--cache", str(scratch / "models"), "--output", str(workloads),\n'
                  '                    "--requests", str(config["requests"]),\n'
                  '                    "--warmup-requests", str(warmup["requests"]),\n'
                  '                    "--warmup-input-tokens", str(warmup["input_tokens"]),\n'
                  '                    "--warmup-output-tokens", str(warmup["output_tokens"])], output / '
                  '"prepare.log", env)\n'
                  '    except Exception as error:\n'
                  '        errors.append(dict(stage="prepare", error=str(error)))\n'
                  '        (output / "errors.json").write_text(json.dumps(errors, indent=2) + "\\n")\n'
                  '        report_fn(output)\n'
                  '        raise\n'
                  '    manifest = json.loads((workloads / "model.json").read_text())\n'
                  '    data = {p.stem: [json.loads(line) for line in p.read_text().splitlines()]\n'
                  '            for p in workloads.glob("*.jsonl")}\n'
                  '    unavailable = set()\n'
                  '    for repeat in range(config["repeats"]):\n'
                  '        engine_order = list(engines)\n'
                  '        random.Random(config["seed"] + repeat).shuffle(engine_order)\n'
                  '        for engine in engine_order:\n'
                  '            arms = list(config["arms"])\n'
                  '            random.Random(config["seed"] + repeat).shuffle(arms)\n'
                  '            for arm in arms:\n'
                  '                if (engine, arm) in unavailable:\n'
                  '                    continue\n'
                  '                run_dir = output / engine / f"r{repeat}-{arm}"\n'
                  '                run_dir.mkdir()\n'
                  '                argv = server_command(engine, engines[engine], manifest["snapshot_path"], '
                  'config, arm, 8000)\n'
                  '                launch = dict(argv=argv, cuda_visible_devices=config["gpu_devices"],\n'
                  '                              model_revision=manifest["resolved_revision"], arm=arm, '
                  'repeat=repeat,\n'
                  '                              dtype="float16", status="STARTING")\n'
                  '                (run_dir / "launch.json").write_text(json.dumps(launch, indent=2) + '
                  '"\\n")\n'
                  '                print(f"Starting {engine} / {arm} / repeat {repeat + 1}", flush=True)\n'
                  '                process = None\n'
                  '                try:\n'
                  '                    with (run_dir / "server.log").open("w") as log:\n'
                  '                        process = subprocess.Popen(argv, stdout=log, '
                  'stderr=subprocess.STDOUT,\n'
                  '                                                   env=env, start_new_session=True)\n'
                  '                        wait_for_server(process, "http://127.0.0.1:8000/v1")\n'
                  '                        smoke_fn("http://127.0.0.1:8000/v1", "qwen-lab", run_dir / '
                  '"smoke.json")\n'
                  '                        for kind, concurrency in campaign_cells(arm, config):\n'
                  '                            summary = benchmark_fn("http://127.0.0.1:8000/v1", '
                  '"qwen-lab", data[kind],\n'
                  '                                concurrency, run_dir / f"{kind}-c{concurrency}",\n'
                  '                                labels=dict(engine=engine, arm=arm, repeat=repeat, '
                  'workload=kind),\n'
                  '                                warmup=data["warmup"], server_pid=process.pid, '
                  'seed=config["seed"],\n'
                  '                                slo=config["slo"])\n'
                  '                            print(f"  {kind} C={concurrency}: '
                  '{summary[\'output_tokens_per_second\']:.1f} tok/s, "\n'
                  '                                  f"errors={summary[\'error_rate\']:.1%}", flush=True)\n'
                  '                            if summary["error_rate"]:\n'
                  '                                raise RuntimeError("Benchmark errors: inspect '
                  'requests.jsonl; comparison invalid")\n'
                  '                    launch["status"] = "MEASURED"\n'
                  '                except Exception as error:\n'
                  '                    launch["status"] = "FAILED"\n'
                  '                    errors.append(dict(engine=engine, arm=arm, repeat=repeat, '
                  'stage="serve/benchmark", error=str(error)))\n'
                  '                    print(f"FAILED {engine}/{arm}: {error}", flush=True)\n'
                  '                    if (run_dir / "server.log").exists():\n'
                  '                        print((run_dir / '
                  '"server.log").read_text(errors="replace")[-7000:], flush=True)\n'
                  '                    if process is not None and process.poll() is not None:\n'
                  '                        # A deterministic startup failure is not retried on identical '
                  'arms.\n'
                  '                        unavailable.add((engine, arm))\n'
                  '                    if config.get("abort_on_server_failure"):\n'
                  '                        raise\n'
                  '                finally:\n'
                  '                    if process is not None:\n'
                  '                        stop_server(process)\n'
                  '                    launch["cgroup_memory_after"] = {}\n'
                  '                    for name in ("memory.events", "memory.peak", "memory.current"):\n'
                  '                        path = Path("/sys/fs/cgroup") / name\n'
                  '                        if path.is_file():\n'
                  '                            launch["cgroup_memory_after"][name] = '
                  'path.read_text().strip()\n'
                  '                    (run_dir / "launch.json").write_text(json.dumps(launch, indent=2) + '
                  '"\\n")\n'
                  '                    (output / "errors.json").write_text(json.dumps(errors, indent=2) + '
                  '"\\n")\n'
                  '                    report_fn(output)\n'
                  '    report_fn(output)\n'
                  '    return output\n'
                  '\n'
                  '\n'
                  'if __name__ == "__main__":\n'
                  '    parser = argparse.ArgumentParser(description=__doc__)\n'
                  '    parser.add_argument("--output", type=Path, required=True)\n'
                  '    parser.add_argument("--config", type=Path)\n'
                  '    args = parser.parse_args()\n'
                  '    sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "benchmarks"))\n'
                  '    from harness import run_benchmark, smoke_server\n'
                  '    from summarize import build_report\n'
                  '    config = json.loads(args.config.read_text()) if args.config else DEFAULT_CONFIG\n'
                  '    run_campaign(config, args.output, run_benchmark, smoke_server, build_report, '
                  'Path(__file__).parent)\n',
 'summarize.py': '"""Build a report only from actual per-request measurements and paired repeats."""\n'
                 'import argparse\n'
                 'import csv\n'
                 'import json\n'
                 'import statistics\n'
                 'from collections import defaultdict\n'
                 'from pathlib import Path\n'
                 '\n'
                 '\n'
                 'def build_report(output):\n'
                 '    output = Path(output)\n'
                 '    groups = defaultdict(list)\n'
                 '    rows = []\n'
                 '    for path in sorted(output.glob("*/r*-*/*/summary.json")):\n'
                 '        row = json.loads(path.read_text())\n'
                 '        row["artifact"] = str(path.relative_to(output))\n'
                 '        rows.append(row)\n'
                 '        if row["error_rate"] == 0:\n'
                 '            groups[(row["engine"], row["arm"], row["workload"], '
                 'row["concurrency"])].append(row)\n'
                 '    config_path = output / "campaign.json"\n'
                 '    config = json.loads(config_path.read_text()) if config_path.exists() else {}\n'
                 '    errors_path = output / "errors.json"\n'
                 '    errors = json.loads(errors_path.read_text()) if errors_path.exists() else []\n'
                 '    table = ["| Engine | Arm | Workload | C | Repeats | Output tok/s | TTFT p50 ms | TPOT '
                 'p50 ms | E2E p95 ms |",\n'
                 '             "|---|---|---|---:|---:|---:|---:|---:|---:|"]\n'
                 '    aggregates = []\n'
                 '\n'
                 '    def median(items, key, quantile=None):\n'
                 '        values = [(item[key][quantile] if quantile else item[key]) for item in items]\n'
                 '        values = [value for value in values if value is not None]\n'
                 '        return statistics.median(values) if values else None\n'
                 '\n'
                 '    def number(value):\n'
                 '        return f"{value:.2f}" if value is not None else "N/A"\n'
                 '\n'
                 '    for (engine, arm, workload, concurrency), items in sorted(groups.items()):\n'
                 '        agg = dict(engine=engine, arm=arm, workload=workload, concurrency=concurrency,\n'
                 '                   repeats=len(items), output_tps=median(items, '
                 '"output_tokens_per_second"),\n'
                 '                   ttft_p50_ms=median(items, "ttft_ms", "p50"),\n'
                 '                   tpot_p50_ms=median(items, "tpot_ms", "p50"),\n'
                 '                   e2e_p95_ms=median(items, "e2e_ms", "p95"))\n'
                 '        aggregates.append(agg)\n'
                 '        table.append(f"| {engine} | {arm} | {workload} | {concurrency} | {len(items)} | " '
                 '+\n'
                 '                     " | ".join(number(agg[key]) for key in ("output_tps", "ttft_p50_ms", '
                 '"tpot_p50_ms", "e2e_p95_ms")) + " |")\n'
                 '    (output / "summaries.json").write_text(json.dumps(rows, indent=2) + "\\n")\n'
                 '    if aggregates:\n'
                 '        with (output / "summary.csv").open("w", newline="") as handle:\n'
                 '            writer = csv.DictWriter(handle, fieldnames=list(aggregates[0]))\n'
                 '            writer.writeheader()\n'
                 '            writer.writerows(aggregates)\n'
                 '    comparisons = []\n'
                 '    for engine in config.get("engines", {}):\n'
                 '        for base, arm, workload, concurrency in (\n'
                 '                ("baseline", "chunked4096", "mixed", 8),\n'
                 '                ("chunked4096", "chunked512", "mixed", 8),\n'
                 '                ("baseline", "prefix", "shared", 8)):\n'
                 '            before = {row["repeat"]: row for row in groups.get((engine, base, workload, '
                 'concurrency), [])}\n'
                 '            after = {row["repeat"]: row for row in groups.get((engine, arm, workload, '
                 'concurrency), [])}\n'
                 '            paired = sorted(before.keys() & after.keys())\n'
                 '            if len(paired) < 3:\n'
                 '                continue\n'
                 '            throughput = [100 * (after[i]["output_tokens_per_second"] / '
                 'before[i]["output_tokens_per_second"] - 1) for i in paired]\n'
                 '            latency = [100 * (after[i]["e2e_ms"]["p95"] / before[i]["e2e_ms"]["p95"] - 1) '
                 'for i in paired]\n'
                 '            ttft = [100 * (after[i]["ttft_ms"]["p50"] / before[i]["ttft_ms"]["p50"] - 1) '
                 'for i in paired]\n'
                 '            comparisons.append(dict(engine=engine, baseline=base, arm=arm, '
                 'workload=workload,\n'
                 '                                    concurrency=concurrency, paired_repeats=len(paired),\n'
                 '                                    output_tps_change_pct=statistics.median(throughput),\n'
                 '                                    output_tps_change_range_pct=[min(throughput), '
                 'max(throughput)],\n'
                 '                                    e2e_p95_change_pct=statistics.median(latency),\n'
                 '                                    ttft_p50_change_pct=statistics.median(ttft)))\n'
                 '    (output / "comparisons.json").write_text(json.dumps(comparisons, indent=2) + "\\n")\n'
                 '    focused = config.get("measurement_scope") == "prefix_validation"\n'
                 '    baseline_cells = config.get("cells", {}).get("baseline") or [\n'
                 '        (kind, c) for kind in ("short", "standard", "long") for c in '
                 'config.get("concurrency", [1, 2, 4, 8, 16])]\n'
                 '    required_baselines = [(engine, "baseline", kind, c) for engine in ("vllm", "sglang")\n'
                 '                          for kind, c in baseline_cells]\n'
                 '    both = all(len(groups.get(key, [])) >= 3 for key in required_baselines)\n'
                 '    status = "MEASURED — see scope and repeat counts" if both else "INCOMPLETE — no '
                 'validated two-engine baseline yet"\n'
                 '    report = f"# Inference optimization report\\n\\nStatus: **{status}**.\\n\\n"\n'
                 '    report += f"Model: `{config.get(\'model\', \'unknown\')}`; FP16; '
                 "CUDA_VISIBLE_DEVICES={config.get('gpu_devices', '?')}; TP={config.get('tensor_parallel', "
                 '\'?\')}.\\n\\n"\n'
                 '    report += f"Requests per cell: {config.get(\'requests\', \'unknown\')}; planned '
                 'independent repeats: {config.get(\'repeats\', \'unknown\')}.\\n\\n"\n'
                 '    shapes = ("2048-token inputs and forced 256-token outputs" if focused else\n'
                 '              "exact templated inputs 128/512/2048 tokens and forced outputs 64/128/256")\n'
                 '    report += (f"Synthetic closed-loop chat load, {shapes}. "\n'
                 '               "Identical model snapshot and workload files are reused by both engines. '
                 'Environments are separate; "\n'
                 '               "this compares packaged serving stacks, including their PyTorch/tokenizer '
                 'versions.\\n\\n"\n'
                 '               "Servers restart for every arm/repeat. Startup, semantic smoke and warmup '
                 'are excluded. "\n'
                 '               "Tables show the median across per-repeat metrics, not a pooled percentile. '
                 '"\n'
                 '               "Tail percentiles are descriptive, without a production p99 or '
                 'production-scale claim.\\n\\n")\n'
                 '    warmup = config.get("warmup", dict(requests=4, input_tokens=512, output_tokens=64))\n'
                 '    report += (f"Warmup: {warmup[\'requests\']} separate '
                 '{warmup[\'input_tokens\']}-input/"\n'
                 '               f"{warmup[\'output_tokens\']}-output requests per point. CUDA graphs are '
                 'disabled.\\n\\n")\n'
                 '    if focused:\n'
                 '        report += ("Focused validation: only shared-prefix 2048→256 at concurrency=8 is '
                 'measured. "\n'
                 '                   "Warmup matches these lengths and covers two full concurrency waves. '
                 'Warmup IDs occur early "\n'
                 '                   "in the prompt and differ from measured prompts, leaving the measured '
                 'shared context unprimed "\n'
                 '                   "apart from the common chat header. Both engines run sequentially on '
                 'one physical GPU. "\n'
                 '                   "64 requests give eight concurrency waves; no production p99 claim is '
                 'made.\\n\\n")\n'
                 '    else:\n'
                 '        report += "The pilot does not establish exhaustive shape/JIT warmup; results are '
                 'exploratory eager serving measurements.\\n\\n"\n'
                 '    report += "\\n".join(table) + "\\n\\n## Paired ablations\\n\\n"\n'
                 '    report += "Relative changes use matched repeats; a negative latency change means lower '
                 'latency.\\n\\n"\n'
                 '    for comparison in comparisons:\n'
                 '        report += f"- `{comparison[\'engine\']}` {comparison[\'baseline\']} → '
                 '{comparison[\'arm\']}, {comparison[\'workload\']} C=8: "\n'
                 '        report += f"output throughput {comparison[\'output_tps_change_pct\']:+.1f}%, E2E '
                 'p95 {comparison[\'e2e_p95_change_pct\']:+.1f}%, "\n'
                 '        report += f"TTFT p50 {comparison[\'ttft_p50_change_pct\']:+.1f}%; '
                 '{comparison[\'paired_repeats\']} paired repeats.\\n"\n'
                 '    if not comparisons:\n'
                 '        report += "No three-repeat ablation result available.\\n"\n'
                 '    report += ("\\n## Measurement boundaries\\n\\nClient TTFT = first non-empty text SSE '
                 'delta. TPOT is a client estimate "\n'
                 '               "from first/last text event and actual server completion_tokens. Token ITL '
                 'is unavailable over "\n'
                 '               "text-only SSE; raw inter-chunk intervals are stored separately. Errors '
                 'contribute to wall time "\n'
                 '               "and error rate, but not successful token throughput. SLO goodput uses the '
                 'fixed campaign thresholds.\\n\\n"\n'
                 '               "VRAM is sampled per physical GPU by nvidia-smi; allocated memory includes '
                 'reserved engine pools. "\n'
                 '               "RAM RSS sums may double-count shared pages. Raw Prometheus snapshots are '
                 'included; no peak KV "\n'
                 '               "occupancy is inferred from endpoint snapshots.\\n\\n"\n'
                 '               "Correctness gates cover semantic smoke, API/stream completion and actual '
                 'prompt/output token counts. "\n'
                 '               "These synthetic load tests do not establish general answer quality or '
                 'lossless decoded-output parity.\\n\\n"\n'
                 '               "Continuous batching remains active. Concurrency and heterogeneous lengths '
                 'test scheduler behavior; "\n'
                 '               "no scheduling-policy speedup is claimed. Speculative decoding and TP=2 are '
                 'NOT EXECUTED in this "\n'
                 '               "first campaign. BF16/FP8 are not tested on T4. Diffusion/video and '
                 'external workers are separate tracks.\\n\\n")\n'
                 '    report += f"Recorded failures in selected measurement campaigns: {len(errors)}. See '
                 '`errors.json`, per-engine `install.log`, `freeze.txt`, `environment.json`, and per-arm '
                 '`server.log`.\\n"\n'
                 '    if config.get("separate_kaggle_sessions"):\n'
                 '        report += f"Earlier failed attempts retained in provenance: '
                 "{len(config.get('historical_error_records', []))}. These are not successful timed "
                 'requests.\\n"\n'
                 '        report += ("\\n## Separate-session provenance\\n\\n"\n'
                 '                   "vLLM and SGLang were measured in separate Kaggle sessions. '
                 'Model/weight hashes, tokenizer/chat "\n'
                 '                   "template, workloads, transport code and campaign parameters were '
                 'validated before merging. "\n'
                 '                   "Within-engine ablations use matched independent repeats. Cross-engine '
                 'comparisons describe "\n'
                 '                   "complete stacks on separate T4 allocations, not paired runs on one '
                 'physical GPU. Original "\n'
                 '                   "sources/configurations and earlier failures are retained under '
                 'provenance/. SGLang\'s JIT link "\n'
                 '                   "fix exposes the existing driver library through a scratch link, '
                 'without changing CUDA kernels.\\n")\n'
                 '    (output / "INFERENCE_OPTIMIZATION_REPORT.md").write_text(report)\n'
                 '    recruiter = "# Ответ рекрутеру — только по выполненным измерениям\\n\\n"\n'
                 '    expected_effects = 2 * sum(arm in config.get("arms", []) for arm in ("chunked4096", '
                 '"chunked512", "prefix"))\n'
                 '    if not both or not expected_effects or len(comparisons) < expected_effects:\n'
                 '        recruiter += ("Ответ с численным приростом пока не готов: нужны baseline обоих '
                 'движков и три повтора "\n'
                 '                      "контролируемых ablations. Подготовка стенда и transport-тесты не '
                 'подтверждают практический "\n'
                 '                      "результат инференса. См. технический отчёт и логи.\\n")\n'
                 '    else:\n'
                 '        recruiter += (f"В учебном R&D-проекте развернул vLLM '
                 '{config[\'engines\'][\'vllm\']} и SGLang {config[\'engines\'][\'sglang\']} "\n'
                 '                      f"с {config[\'model\']} в FP16 на одной NVIDIA T4. Настроил '
                 'OpenAI-compatible streaming и "\n'
                 '                      "non-streaming chat serving; написал health/correctness checks и '
                 'общий closed-loop benchmark. Проверял "\n'
                 '                      + ("shared-prefix запросы 2048→256, concurrency=8; по 64 запроса и '
                 'три перезапуска, "\n'
                 '                         "16 warmup той же длины. Оба движка последовательно на одной GPU. '
                 'Это лабораторный опыт.\\n\\n"\n'
                 '                         if focused else f"короткие, длинные и смешанные запросы при '
                 'concurrency {config.get(\'concurrency\')}. Это лабораторный опыт.\\n\\n")\n'
                 '                      + ("В этой проверке менял только prefix caching ON/OFF. " if focused '
                 'else\n'
                 '                         "Настраивал chunked prefill и его token budget, prefix caching; '
                 'отдельно исследовал поведение continuous batching при неоднородных длинах. ")\n'
                 '                      + "Ниже медианы относительных изменений по трём или более повторам '
                 'на заранее выбранных сценариях; фактические tokens взяты из server usage.\\n\\n")\n'
                 '        for comparison in comparisons:\n'
                 '            lo, hi = comparison["output_tps_change_range_pct"]\n'
                 '            recruiter += (f"- {comparison[\'engine\']}, {comparison[\'baseline\']} → '
                 '{comparison[\'arm\']}, "\n'
                 '                          f"{comparison[\'workload\']}, C=8: throughput '
                 '{comparison[\'output_tps_change_pct\']:+.1f}% "\n'
                 '                          f"(диапазон повторов {lo:+.1f}…{hi:+.1f}%), E2E p95 '
                 '{comparison[\'e2e_p95_change_pct\']:+.1f}%, "\n'
                 '                          f"TTFT p50 {comparison[\'ttft_p50_change_pct\']:+.1f}%.\\n")\n'
                 '        recruiter += ("\\nСценарий shared использует общий длинный prefix; эти результаты '
                 'нельзя переносить на "\n'
                 '                      "произвольный чат. Политики scheduler, speculative decoding и '
                 'multi-GPU scaling в этот "\n'
                 '                      "прогон не входили. Конфигурации, per-request JSONL, ошибки, замеры '
                 'памяти и версии сохранены.\\n")\n'
                 '    (output / "RECRUITER_ANSWER_RU.md").write_text(recruiter)\n'
                 '    build_scheduling_report(output)\n'
                 '    if aggregates:\n'
                 '        if focused:\n'
                 '            plot_prefix_validation(output, aggregates)\n'
                 '        else:\n'
                 '            plot_results(output, aggregates)\n'
                 '    return dict(status=status, measured_cells=len(rows), comparisons=comparisons, '
                 'errors=len(errors))\n'
                 '\n'
                 '\n'
                 'def build_scheduling_report(output):\n'
                 '    """Compare the same short request IDs alone and beside long requests."""\n'
                 '    cells = []\n'
                 '    short_ids = {}\n'
                 '    for path in output.glob("*/r*-*/*/summary.json"):\n'
                 '        summary = json.loads(path.read_text())\n'
                 '        requests_path = path.parent / "requests.jsonl"\n'
                 '        if summary["error_rate"] or summary["workload"] not in ("short", "mixed") or not '
                 'requests_path.exists():\n'
                 '            continue\n'
                 '        if summary["concurrency"] not in (8, 16):\n'
                 '            continue\n'
                 '        requests = [json.loads(line) for line in requests_path.read_text().splitlines()]\n'
                 '        cells.append((summary, requests))\n'
                 '        if summary["arm"] == "baseline" and summary["workload"] == "mixed":\n'
                 '            key = (summary["engine"], summary["repeat"], summary["concurrency"])\n'
                 '            short_ids[key] = {row["id"].rsplit("-", 1)[-1] for row in requests if '
                 'row["expected_prompt_tokens"] == 128}\n'
                 '    buckets = []\n'
                 '    for summary, requests in cells:\n'
                 '        cases = []\n'
                 '        if summary["workload"] == "mixed":\n'
                 '            cases = [("mixed-short", [row for row in requests if '
                 'row["expected_prompt_tokens"] == 128]),\n'
                 '                     ("mixed-long", [row for row in requests if '
                 'row["expected_prompt_tokens"] == 2048])]\n'
                 '        elif summary["arm"] == "baseline":\n'
                 '            ids = short_ids.get((summary["engine"], summary["repeat"], '
                 'summary["concurrency"]))\n'
                 '            if ids:\n'
                 '                cases = [("homogeneous-short", [row for row in requests if '
                 'row["id"].rsplit("-", 1)[-1] in ids])]\n'
                 '        for case, selected in cases:\n'
                 '            if not selected:\n'
                 '                continue\n'
                 '            bucket = {key: summary[key] for key in ("engine", "arm", "repeat", '
                 '"concurrency")}\n'
                 '            bucket.update(case=case, n=len(selected))\n'
                 '            for metric in ("ttft_ms", "tpot_ms", "e2e_ms"):\n'
                 '                values = [row[metric] for row in selected if row[metric] is not None]\n'
                 '                bucket[metric + "_p50"] = statistics.median(values) if values else None\n'
                 '                bucket[metric + "_p95"] = (statistics.quantiles(values, n=100, '
                 'method="inclusive")[94]\n'
                 '                                             if len(values) > 1 else (values[0] if values '
                 'else None))\n'
                 '            buckets.append(bucket)\n'
                 '    if not buckets:\n'
                 '        return []\n'
                 '    with (output / "scheduling.csv").open("w", newline="") as handle:\n'
                 '        writer = csv.DictWriter(handle, fieldnames=list(buckets[0]))\n'
                 '        writer.writeheader()\n'
                 '        writer.writerows(buckets)\n'
                 '    comparisons = []\n'
                 '    for engine in ("vllm", "sglang"):\n'
                 '        for base, arm in (("baseline", "chunked4096"), ("chunked4096", "chunked512")):\n'
                 '            for concurrency in (8, 16):\n'
                 '                before = {r["repeat"]: r for r in buckets if r["engine"] == engine and '
                 'r["arm"] == base\n'
                 '                          and r["case"] == "mixed-short" and r["concurrency"] == '
                 'concurrency}\n'
                 '                after = {r["repeat"]: r for r in buckets if r["engine"] == engine and '
                 'r["arm"] == arm\n'
                 '                         and r["case"] == "mixed-short" and r["concurrency"] == '
                 'concurrency}\n'
                 '                paired = sorted(before.keys() & after.keys())\n'
                 '                if len(paired) < 3:\n'
                 '                    continue\n'
                 '                changes = [100 * (after[i]["ttft_ms_p50"] / before[i]["ttft_ms_p50"] - 1) '
                 'for i in paired]\n'
                 '                comparisons.append(dict(engine=engine, baseline=base, arm=arm, '
                 'concurrency=concurrency,\n'
                 '                                        paired_repeats=len(paired), case="short requests '
                 'inside mixed load",\n'
                 '                                        ttft_p50_change_pct=statistics.median(changes),\n'
                 '                                        ttft_change_range_pct=[min(changes), '
                 'max(changes)]))\n'
                 '    (output / "scheduling-comparisons.json").write_text(json.dumps(comparisons, indent=2) '
                 '+ "\\n")\n'
                 '    report = ("# Scheduling / heterogeneous load\\n\\n"\n'
                 '              "`scheduling.csv` separates short and long requests. Homogeneous short '
                 'latency uses the same request "\n'
                 '              "IDs as the short half of the mixed workload, with the same 128-token input '
                 'and 64-token output. "\n'
                 '              "This measures the effect of competing long requests under closed-loop load. '
                 'It does not isolate "\n'
                 '              "scheduler CPU overhead from prefill GPU work or prove a specific '
                 'head-of-line mechanism. "\n'
                 '              "No scheduling policy is changed. Each short bucket has only 16 requests per '
                 'pilot repeat; "\n'
                 '              "tail estimates are exploratory.\\n\\n"\n'
                 '              "## Paired chunking effects on short-request TTFT\\n\\n")\n'
                 '    for item in comparisons:\n'
                 '        report += (f"- {item[\'engine\']}, {item[\'baseline\']} → {item[\'arm\']}, '
                 'C={item[\'concurrency\']}: "\n'
                 '                   f"short-request TTFT p50 {item[\'ttft_p50_change_pct\']:+.1f}%, "\n'
                 '                   f"{item[\'paired_repeats\']} paired repeats.\\n")\n'
                 '    if not comparisons:\n'
                 '        report += "No three-repeat short-request comparison available yet.\\n"\n'
                 '    (output / "SCHEDULING_REPORT.md").write_text(report)\n'
                 '    return comparisons\n'
                 '\n'
                 '\n'
                 'def plot_prefix_validation(output, aggregates):\n'
                 '    import matplotlib\n'
                 '    matplotlib.use("Agg")\n'
                 '    import matplotlib.pyplot as plt\n'
                 '    plots = output / "plots"\n'
                 '    plots.mkdir(exist_ok=True)\n'
                 '    rows = json.loads((output / "summaries.json").read_text())\n'
                 '    config = json.loads((output / "campaign.json").read_text())\n'
                 '    fig, axes = plt.subplots(2, 2, figsize=(10, 7.5))\n'
                 '    for index, engine in enumerate(("vllm", "sglang")):\n'
                 '        for column, key, label in ((0, "output_tps", "Output tokens/s"), (1, '
                 '"ttft_p50_ms", "Client TTFT p50, ms")):\n'
                 '            axis = axes[index, column]\n'
                 '            raw = {}\n'
                 '            for x, arm in enumerate(("baseline", "prefix")):\n'
                 '                selected = next((r for r in aggregates if r["engine"] == engine and '
                 'r["arm"] == arm), None)\n'
                 '                if selected:\n'
                 '                    axis.bar(x, selected[key], width=0.55, color=("#64748b", '
                 '"#0891b2")[x])\n'
                 '                    raw[arm] = {r["repeat"]: r["output_tokens_per_second"] if column == 0 '
                 'else r["ttft_ms"]["p50"]\n'
                 '                                for r in rows if r["engine"] == engine and r["arm"] == '
                 'arm}\n'
                 '                    axis.scatter([x] * len(raw[arm]), list(raw[arm].values()), s=22,\n'
                 '                                 color="#0f172a", edgecolors="white", linewidths=0.6, '
                 'zorder=4)\n'
                 '                    axis.annotate(f"{selected[key]:,.1f}", (x, selected[key]), xytext=(0, '
                 '9),\n'
                 '                                  textcoords="offset points", ha="center", fontsize=9)\n'
                 '            for repeat in raw.get("baseline", {}).keys() & raw.get("prefix", {}).keys():\n'
                 '                axis.plot([0, 1], [raw["baseline"][repeat], raw["prefix"][repeat]],\n'
                 '                          color="#334155", alpha=0.35, linewidth=0.8, zorder=3)\n'
                 '            axis.set(xticks=[0, 1], xticklabels=["Cache off", "Cache on"], ylabel=label,\n'
                 '                     title=f"{engine}: shared prefix, C=8")\n'
                 '            axis.margins(y=0.18)\n'
                 '            axis.grid(axis="y", alpha=0.2)\n'
                 '    fig.suptitle(f"Qwen2.5-0.5B / T4 / FP16 eager / 2048 input + 256 output\\n"\n'
                 '                 f"{config[\'requests\']} requests per cell; {config[\'repeats\']} server '
                 'restarts per arm", fontsize=11)\n'
                 '    fig.text(0.5, 0.015, "Bars: medians of per-restart metrics. Dots/lines: individual '
                 'matched restarts.",\n'
                 '             ha="center", fontsize=9, color="#334155")\n'
                 '    fig.tight_layout(rect=(0, 0.035, 1, 0.93))\n'
                 '    fig.savefig(plots / "prefix-validation.png", dpi=160)\n'
                 '    plt.close(fig)\n'
                 '\n'
                 '\n'
                 'def plot_results(output, aggregates):\n'
                 '    try:\n'
                 '        import matplotlib\n'
                 '        matplotlib.use("Agg")\n'
                 '        import matplotlib.pyplot as plt\n'
                 '    except ImportError:\n'
                 '        (output / "PLOTS_UNAVAILABLE.txt").write_text("Install matplotlib in an isolated '
                 'reporting environment.\\n")\n'
                 '        return\n'
                 '    plots = output / "plots"\n'
                 '    plots.mkdir(exist_ok=True)\n'
                 '    fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))\n'
                 '    for axis, kind in zip(axes, ("short", "standard", "long")):\n'
                 '        for engine in ("vllm", "sglang"):\n'
                 '            selected = sorted([row for row in aggregates if row["arm"] == "baseline" and '
                 'row["workload"] == kind\n'
                 '                               and row["engine"] == engine], key=lambda row: '
                 'row["concurrency"])\n'
                 '            axis.plot([r["concurrency"] for r in selected], [r["output_tps"] for r in '
                 'selected], "o-", label=engine)\n'
                 '        axis.set(title=kind, xlabel="Concurrency", ylabel="Output tokens/s")\n'
                 '        axis.grid(alpha=0.25)\n'
                 '        axis.legend()\n'
                 '    fig.tight_layout()\n'
                 '    fig.savefig(plots / "throughput-concurrency.png", dpi=160)\n'
                 '    plt.close(fig)\n'
                 '    fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))\n'
                 '    for axis, key, label in zip(axes, ("ttft_p50_ms", "tpot_p50_ms", "e2e_p95_ms"),\n'
                 '                               ("TTFT p50, ms", "TPOT p50 estimate, ms", "E2E p95, ms")):\n'
                 '        for engine in ("vllm", "sglang"):\n'
                 '            selected = sorted([row for row in aggregates if row["arm"] == "baseline" and '
                 'row["workload"] == "long"\n'
                 '                               and row["engine"] == engine], key=lambda row: '
                 'row["concurrency"])\n'
                 '            axis.plot([r["concurrency"] for r in selected], [r[key] for r in selected], '
                 '"o-", label=engine)\n'
                 '        axis.set(xlabel="Concurrency", ylabel=label, title="Long context")\n'
                 '        axis.grid(alpha=0.25)\n'
                 '        axis.legend()\n'
                 '    fig.tight_layout()\n'
                 '    fig.savefig(plots / "latency-concurrency.png", dpi=160)\n'
                 '    plt.close(fig)\n'
                 '    fig, axis = plt.subplots(figsize=(6, 4))\n'
                 '    for engine in ("vllm", "sglang"):\n'
                 '        selected = sorted([row for row in aggregates if row["arm"] == "baseline" and '
                 'row["workload"] == "long"\n'
                 '                           and row["engine"] == engine], key=lambda row: '
                 'row["concurrency"])\n'
                 '        axis.plot([r["output_tps"] for r in selected], [r["e2e_p95_ms"] for r in '
                 'selected], "o-", label=engine)\n'
                 '    axis.set(xlabel="Output tokens/s", ylabel="E2E p95, ms", title="Latency / throughput: '
                 'long context")\n'
                 '    axis.grid(alpha=0.25)\n'
                 '    axis.legend()\n'
                 '    fig.tight_layout()\n'
                 '    fig.savefig(plots / "latency-throughput.png", dpi=160)\n'
                 '    plt.close(fig)\n'
                 '    fig, axes = plt.subplots(2, 2, figsize=(11, 7))\n'
                 '    for row_index, workload, arms in ((0, "mixed", ("baseline", "chunked4096", '
                 '"chunked512")),\n'
                 '                                      (1, "shared", ("baseline", "prefix"))):\n'
                 '        for column, metric, ylabel in ((0, "output_tps", "Output tokens/s"),\n'
                 '                                       (1, "e2e_p95_ms", "E2E p95, ms")):\n'
                 '            axis = axes[row_index, column]\n'
                 '            for index, engine in enumerate(("vllm", "sglang")):\n'
                 '                selected = {r["arm"]: r for r in aggregates if r["engine"] == engine\n'
                 '                            and r["workload"] == workload and r["concurrency"] == 8}\n'
                 '                present = [(i, arm) for i, arm in enumerate(arms) if arm in selected]\n'
                 '                axis.bar([i + (index - 0.5) * 0.36 for i, _ in present],\n'
                 '                         [selected[arm][metric] for _, arm in present], width=0.36, '
                 'label=engine)\n'
                 '            axis.set(xticks=list(range(len(arms))), xticklabels=arms, ylabel=ylabel,\n'
                 '                     title=f"{workload}, concurrency=8")\n'
                 '            axis.grid(axis="y", alpha=0.2)\n'
                 '            axis.legend()\n'
                 '    fig.tight_layout()\n'
                 '    fig.savefig(plots / "baseline-ablations.png", dpi=160)\n'
                 '    plt.close(fig)\n'
                 '    memory = defaultdict(list)\n'
                 '    for summary_path in output.glob("*/r*-baseline/long-c*/summary.json"):\n'
                 '        summary = json.loads(summary_path.read_text())\n'
                 '        resource_path = summary_path.parent / "resources.jsonl"\n'
                 '        if not resource_path.exists():\n'
                 '            continue\n'
                 '        peaks = {}\n'
                 '        for line in resource_path.read_text().splitlines():\n'
                 '            for gpu in json.loads(line).get("gpu", []):\n'
                 '                try:\n'
                 '                    value = float(gpu["memory_used_mib"])\n'
                 '                except (ValueError, KeyError):\n'
                 '                    continue\n'
                 '                peaks[gpu["index"]] = max(peaks.get(gpu["index"], 0), value)\n'
                 '        for index, peak in peaks.items():\n'
                 '            memory[(summary["engine"], summary["concurrency"], index)].append(peak)\n'
                 '    if memory:\n'
                 '        fig, axis = plt.subplots(figsize=(7, 4))\n'
                 '        for engine, index in sorted({(engine, index) for engine, _, index in memory}):\n'
                 '            concurrencies = sorted(c for e, c, i in memory if e == engine and i == index)\n'
                 '            axis.plot(concurrencies, [statistics.median(memory[(engine, c, index)]) for c '
                 'in concurrencies],\n'
                 '                      "o-", label=f"{engine} GPU {index}")\n'
                 '        axis.set(xlabel="Concurrency", ylabel="Sampled peak resident VRAM, MiB",\n'
                 '                 title="Long context: median across repeats, including reserved pools")\n'
                 '        axis.grid(alpha=0.25)\n'
                 '        axis.legend()\n'
                 '        fig.tight_layout()\n'
                 '        fig.savefig(plots / "memory-concurrency.png", dpi=160)\n'
                 '        plt.close(fig)\n'
                 '\n'
                 '\n'
                 'if __name__ == "__main__":\n'
                 '    parser = argparse.ArgumentParser(description=__doc__)\n'
                 '    parser.add_argument("output", type=Path)\n'
                 '    print(json.dumps(build_report(parser.parse_args().output), indent=2))\n'}

SOURCE_DIR = Path('/kaggle/temp/inference-lab-source')
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
for name, source in SOURCES.items():
    (SOURCE_DIR / name).write_text(source)
sys.path.insert(0, str(SOURCE_DIR))
print('Prepared:', ', '.join(SOURCES))


## 3. Параметры кампании

Baseline: prefix cache OFF, chunked prefill OFF, eager, FP16, TP=1, FCFS (SGLang).
Сначала включаем chunked prefill с бюджетом 4096, затем отдельно уменьшаем бюджет до 512.
Prefix cache ON сравнивается с baseline на одинаковых shared-prefix запросах.
Между arm/repeat — новый сервер. Warmup использует другие request IDs и исключён из метрик.

Число запросов в ячейке должно быть не меньше максимальной concurrency. Для репетиций
сохраняются одни и те же входы. `ignore_eos=True` применяется только к synthetic нагрузке;
semantic smoke проверяет естественную остановку и оба режима API.


In [ ]:
import copy
from datetime import datetime, timezone
from harness import run_benchmark, smoke_server
from run_kaggle import DEFAULT_CONFIG, run_campaign
from summarize import build_report

CONFIG = copy.deepcopy(DEFAULT_CONFIG)
# Увеличивать после успешного первого baseline, например до 128/256 requests.
CONFIG['requests'] = 32
CONFIG['repeats'] = 3
OUTPUT = Path('/kaggle/working') / ('inference-results-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
print(json.dumps(CONFIG, indent=2))
print('Output:', OUTPUT)


## 4. Установка → smoke → измерения → отчёт

Оба движка получают один snapshot/tokenizer, его полный revision и SHA-256 файлов записываются.
Серверы доступны только по loopback. Неуспешные установки/запуски сохраняются в logs и `errors.json`;
они не превращаются в нулевую latency или «ускорение» другого движка.

TTFT здесь — первый непустой текстовый delta на клиенте; TPOT — оценка по фактическому числу
generated tokens. **Интервалы SSE не выдаются за token ITL.**


In [ ]:
RESULTS = run_campaign(CONFIG, OUTPUT, run_benchmark, smoke_server, build_report, SOURCE_DIR)
print('Saved:', RESULTS)


## 5. Результаты для технического отчёта и рекрутера

Сохраните Version с Output: `INFERENCE_OPTIMIZATION_REPORT.md`, `RECRUITER_ANSWER_RU.md`,
`summary.csv`, `comparisons.json`, `plots/`, `workloads/`, `source/` и логи всех arms.
Ответ с приростом появится только при наличии трёх повторов и baseline обоих движков.
Speculative decoding, TP=2 и видеогенерация в эту первую кампанию не входят.


In [ ]:
from IPython.display import display, Markdown, Image
display(Markdown((OUTPUT / 'RECRUITER_ANSWER_RU.md').read_text()))
for plot in sorted((OUTPUT / 'plots').glob('*.png')):
    display(Image(filename=str(plot)))
